# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Use CUDA async allocator to reduce fragmentation:
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# Suppress TensorFlow logging (0: ALL, 1: INFO, 2: WARNING, 3: ERROR):
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# # If it fails to determine best cudnn convolution algorithm
# os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [2]:
# # Disable all auto-JIT clustering at the process level
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [3]:
from _imports import * # Centralized file containing all imports

2026-01-07 16:36:57.107890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767814617.126301  770353 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767814617.132210  770353 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


### 1.3. GPU Management

In [4]:
get_gpu_info()

TensorFlow GPU Monitor - 2026-01-07 16:36:58
TensorFlow Configuration
Version        : 2.18.0
CUDA Support: Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 4070      1.1GB /   12.0GB  43C    6%    



I0000 00:00:1767814619.134900  770353 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1767814619.135723  770353 gpu_device.cc:2022] Created device /device:GPU:0 with 9166 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:b3:00.0, compute capability: 8.9


## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 500
EPOCHS = 50

SAMPLER_SEED = 0

STEPS_PER_EXECUTION = 32

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
USE_JIT_COMPILE = True

In [6]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "s009_accuracy"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "minimize"

In [7]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [8]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [9]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels(s008_path="./data/s008", s009_path="./data/s009")

s008_y_train dtype: complex128
s008_y_train min: (4.197915626225068e-11+0j)
s008_y_train max: (0.08270388841629028+0j)
s008_y_train shape: (9234, 8, 32)
First 10 entries of s008_y_train:
[[[1.48147967e-06+0.j 1.42114072e-06+0.j 1.03133857e-06+0.j ...
   1.80768166e-06+0.j 1.63874131e-06+0.j 1.57037084e-06+0.j]
  [1.72811065e-06+0.j 1.65490076e-06+0.j 1.22581332e-06+0.j ...
   2.12785380e-06+0.j 1.92409152e-06+0.j 1.83514169e-06+0.j]
  [2.55523173e-06+0.j 2.44047396e-06+0.j 1.86423688e-06+0.j ...
   3.18969273e-06+0.j 2.87317789e-06+0.j 2.72109719e-06+0.j]
  ...
  [2.82377323e-06+0.j 2.72887883e-06+0.j 1.76969399e-06+0.j ...
   3.29349132e-06+0.j 3.02324702e-06+0.j 2.96806957e-06+0.j]
  [1.80020947e-06+0.j 1.73299827e-06+0.j 1.19801757e-06+0.j ...
   2.15278078e-06+0.j 1.96249243e-06+0.j 1.90079936e-06+0.j]
  [1.49894072e-06+0.j 1.44012438e-06+0.j 1.02391641e-06+0.j ...
   1.81328801e-06+0.j 1.64774929e-06+0.j 1.58619366e-06+0.j]]

 [[3.11911353e-05+0.j 1.19103333e-05+0.j 1.17593581e-05

/home/matheus/src/RayWise/src/_load_dataset.py:54: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:89: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:125: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


## Hyperparameters

In [10]:
kparams = KParams(
    activation_choices={
        "relu": tf.keras.activations.relu,
        "gelu": tf.keras.activations.gelu,
        "silu": tf.keras.activations.silu,
        "elu": tf.keras.activations.elu,
        "sigmoid": tf.keras.activations.sigmoid,
        "tanh": tf.keras.activations.tanh,
        "none": None,
    },
    regularizer_choices={
        # "l2": tf.keras.regularizers.l2,
        "none": None,
    },
    optimizer_choices={
        # "sgd": tf.keras.optimizers.SGD(momentum=0.9),
        # "adam": tf.keras.optimizers.Adam(),
        "adamw": tf.keras.optimizers.AdamW(weight_decay=1e-4),
        # "lion": tf.keras.optimizers.Lion(beta_1=0.9, beta_2=0.99),
        # "rmsprop": tf.keras.optimizers.RMSprop(),
    },
    # scaler_choices={
    #     "standard": StandardScaler,
    #     "minmax_0_1": lambda: MinMaxScaler(feature_range=(0, 1)),
    #     "minmax_-1_1": lambda: MinMaxScaler(feature_range=(-1, 1)),
    # },
    learning_rate=(1e-4, 1e-2),
)

I0000 00:00:1767814620.242213  770353 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9166 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:b3:00.0, compute capability: 8.9


## 5. Model Definition

In [11]:
def build_model(
    trial: optuna.Trial,
    kparams: dict,
    *,
    show_summary: bool = True,
    **kwargs: Any,
) -> tf.keras.Model:

    train_seed = kwargs.get("train_seed")

    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=train_seed,
    )

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Downsample the grid before flattening to cut the sequence length (and FLOPs).
    one_hot_lidar = layers.MaxPooling2D(
        pool_size=(2, 2),
        name="lidar_pool_2d",
    )(one_hot_lidar)

    seq_len = (20 // 2) * (200 // 2)

    # Flatten the pooled grid into a shorter sequence with the 4 channels.
    x_lidar_flat: layers.Layer = layers.Reshape((seq_len, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,seq_len,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, seq_len, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(seq_len, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,seq_len,4) + (batch,seq_len,2) → (batch,seq_len,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 2)

    for i in range(num_conv_layers):
        x = build_cnn1d(
            trial=trial,
            kparams=kparams,
            x=combined if i == 0 else x,  # Use combined only for the first layer
            name_prefix=f"conv1d_{i}",
            # Filters
            filters_range=trial.suggest_categorical(f"conv1d_{i}_filters", [32, 64, 96, 128]),
            # filters_step=40,
            # Kernel size
            kernel_size_range=(1, 9),
            kernel_size_step=1,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
            kernel_initializer=initializer,
        )
        #! pool size = 1 means no downsampling
        pool_size = trial.suggest_int(f"pool_size_{i}", 1, 4, step=1)
        x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)

    pooling_type = trial.suggest_categorical("pooling_type", ["flatten", "max", "average"])
    if pooling_type == "flatten":
        x = layers.Flatten(name="flatten_cnn_output")(x)
    elif pooling_type == "max":
        x = layers.GlobalMaxPooling1D(name="global_max_pooling")(x)
    else:
        x = layers.GlobalAveragePooling1D(name="global_avg_pooling")(x)

    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 1)
    for i in range(num_dense_layers):
        x = build_dnn(
            trial=trial,
            kparams=kparams,
            x=x,
            name_prefix=f"dense_{i}",
            units_range=(25, 200),
            units_step=25,
            dropout_rate_range=(0.0, 0.5),
            dropout_rate_step=0.1,
            kernel_initializer=initializer,
        )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=USE_JIT_COMPILE,  # For XLA speedup, does not support determinism
        steps_per_execution=STEPS_PER_EXECUTION,
    )

    return model

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    **kwargs: Any,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        **kwargs: Additional keyword arguments.

    Returns:
        float: Final value used for optimization.
    """
    (print(f"Running trial {trial.number}..."), clear_session())

    # —————————————————————————————— Reproducibility ————————————————————————————— #
    DATA_SEED = 0
    TRAIN_SEED = 0

    # Set Python, NumPy, Keras and TensorFlow seeds
    set_random_seed(TRAIN_SEED)

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    global s009_coord_input, s009_lidar_input, s009_y
    global s008_coord_input, s008_lidar_input, s008_y_train

    (
        x_s008_lidar_train,
        x_s008_lidar_val,
        x_s008_coord_train,
        x_s008_coord_val,
        y_s008_train,
        y_s008_val,
    ) = train_test_split(
        s008_lidar_input,
        s008_coord_input,
        s008_y_train,
        test_size=0.2,
        random_state=DATA_SEED,
        shuffle=True,
    )

    backup_dir = kwargs["backup"]
    model_dir = kwargs["model"]
    fig_dir = kwargs["fig"]
    tensorboard_dir = kwargs["tensorboard"]
    logs_dir = kwargs["logs"]
    history_dir = kwargs["history"]
    scaler_dir = kwargs["scaler"]

    # ———————————————————————————————————————————————————————————————————————————— #

    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_s008_coord_train)
        x_s008_coord_train = coord_scaler.transform(x_s008_coord_train)
        x_s008_coord_val = coord_scaler.transform(x_s008_coord_val)
        s009_coord_input = coord_scaler.transform(s009_coord_input)
        s008_coord_input = coord_scaler.transform(s008_coord_input)

        scaler_path = os.path.join(scaler_dir, f"trial_{trial.number}.pkl")
        with open(scaler_path, "wb") as scaler_file:
            pickle.dump(coord_scaler, scaler_file)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                        Model Construction and Training                       #
        # ———————————————————————————————————————————————————————————————————————————— #
        model = build_model(
            trial=trial,
            kparams=kparams,
            show_summary=False,
            train_seed=TRAIN_SEED,
        )
        BATCH_SIZE = 64

        prune_model_by_config(
            trial=trial,
            model=model,
            thresholds={
                "model_size": 350,  # Maximum model size in MB
                "memory_mb": 9000,  # Maximum memory training usage in MB
                "param": 0.5e7,  # Maximum number of parameters
                "flops": 28.31e6,  # Maximum number of FLOPs
            },
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
        )

        history = model.fit(
            x=[x_s008_lidar_train, x_s008_coord_train],
            y=y_s008_train,
            validation_data=([x_s008_lidar_val, x_s008_coord_val], y_s008_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks_study(
                trial=trial,
                monitor="val_loss",
                #! Can cause high memory usage
                # tensorboard_logs=tensorboard_dir,
            ),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
            test_runs=10,
            device="gpu/0",
            stats_to_measure=(
                "parameters",
                "model_size",
                "flops",
                "macs",
                "summary",
                "inference_latency",
                # "cpu_util_percent",
                # "cpu_power_rapl_w",
                # "ram_used_bytes",
                # "ram_util_percent",
                # "gpu_util_percent",
                # "gpu_mem_used_bytes",
                # "gpu_power_w",
            ),
            extra_attrs=None,
            verbose=1,
        )

        # Evaluate on full s009
        s009_loss, s009_acc = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=2
        )

        # Evaluate on s008
        s008_loss, s008_acc = model.evaluate(
            [s008_lidar_input, s008_coord_input], s008_y_train, batch_size=BATCH_SIZE, verbose=2
        )

        # ———————————————————————————————————————————————————————————————————————————— #
        #                               Extra Attributes                               #
        # ———————————————————————————————————————————————————————————————————————————— #
        # # Choose best epoch based on validation loss
        # if "minimize" in DIRECTION:
        #     # The best epoch is the one with the lowest validation loss
        #     best_idx = int(np.argmin(history.history["val_loss"]))
        # else:
        #     # The best epoch is the one with the highest validation loss
        #     best_idx = int(np.argmax(history.history["val_loss"]))
            
        # Choose best epoch based on validation accuracy
        best_idx = int(np.argmax(history.history["val_accuracy"]))

        best_train_loss = float(history.history["loss"][best_idx])
        best_val_loss = float(history.history["val_loss"][best_idx])
        best_train_acc = float(history.history["accuracy"][best_idx])
        best_val_acc = float(history.history["val_accuracy"][best_idx])

        trial.set_user_attr("best_epoch", best_idx + 1)
        trial.set_user_attr("best_train_loss", best_train_loss)
        trial.set_user_attr("best_val_loss", best_val_loss)
        trial.set_user_attr("s008_loss", float(s008_loss))
        trial.set_user_attr("s009_loss", float(s009_loss))

        trial.set_user_attr("best_train_accuracy", best_train_acc)
        trial.set_user_attr("best_val_accuracy", best_val_acc)
        trial.set_user_attr("s008_accuracy", float(s008_acc))
        trial.set_user_attr("s009_accuracy", float(s009_acc))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
            "train_accuracy": history.history["accuracy"],
            "val_accuracy": history.history["val_accuracy"],
        }

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(history.history["val_loss"]) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=history.history["val_loss"])

        return best_val_loss  # Value to minimize or maximize
    except ValueError as e:
        # Catch invalid model configurations
        # e.g., when a pooling operation results in negative dimension size
        if "Negative dimension size" in str(e):
            raise optuna.TrialPruned("Pruned, invalid pooling config") from e
        raise
    except Exception as e:
        log_trial_error(
            trial=trial,
            exc=e,
            logs_dir=logs_dir,
            prune_on={
                tf.errors.ResourceExhaustedError: None,
                tf.errors.InternalError: None,
                tf.errors.UnavailableError: None,
            },
            propagate={
                optuna.exceptions.TrialPruned: None,
            },
            force_crash_oom=None,  # Crash after X occurrences of OOM
        )

## Main

In [ ]:
# # Search space:
# base_path = f"{RUN_DIR}/search_space/"
# (
#     x_s008_lidar_train,
#     x_s008_lidar_val,
#     x_s008_coord_train,
#     x_s008_coord_val,
#     y_s008_train,
#     y_s008_val,
# ) = train_test_split(
#     s008_lidar_input,
#     s008_coord_input,
#     s008_y_train,
#     test_size=0.2,
#     random_state=0,
#     shuffle=True,
# )


# plot_model_param_distribution(
#     lambda trial: build_model(
#         trial=trial,
#         kparams=kparams,
#         show_summary=False,
#         train_seed=0,
#     ),
#     benchmark_training=False,
#     fit_x=(x_s008_lidar_train, x_s008_coord_train),
#     fit_y=y_s008_train,
#     fit_validation_data=((x_s008_lidar_val, x_s008_coord_val), y_s008_val),
#     bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
#     batch_size=1,
#     n_trials=NUM_TRIALS,
#     fig_save_path=f"{base_path}model_param_distribution.png",
#     csv_path=f"{base_path}model_param_distribution.csv",
#     logs_dir=f"{base_path}logs/",
#     corr_csv_path=f"{base_path}model_param_distribution_corr.csv",
#     # plot_model_dir=f"{base_path}plots/",
#     figsize=(18, 6),
# )

[I 2026-01-07 16:37:00,410] A new study created in memory with name: no-name-dea56d35-5d6f-4092-a774-4478080cb678
Sampling models: 100% [###########################] 1000/1000 in 00:00


In [ ]:
study = run_study(
    objective=objective,
    run_dir=RUN_DIR,
    num_trials=NUM_TRIALS,
    sampler_seed=SAMPLER_SEED,
    direction=DIRECTION,
    top_k=TOP_K,
    rank_key=RANK_KEY,
    order=ORDER,
    convergence_epoch_column="train_loss",
    convergence_epoch_direction="minimize",
    init_study_dirs=[
        "args",
        "fig",
        "backup",
        "history",
        "scaler",
        "model",
        "logs",
        "tensorboard",
    ],
    cleanup_paths=[
        ("model", "trial_{trial_id}.keras"),
        ("fig", "trial_{trial_id}.png"),
        ("history", "trial_{trial_id}.csv"),
        ("tensorboard", "trial_{trial_id}"),
        ("scaler", "trial_{trial_id}.pkl"),
    ],
    rename_paths=[
        ("model", ".keras"),
        ("fig", ".png"),
        ("history", ".csv"),
        ("scaler", ".pkl"),
    ],
    extra_attrs=[
        "best_epoch",
        "best_train_loss",
        "best_val_loss",
        "s008_loss",
        "s009_loss",
        "best_train_accuracy",
        "best_val_accuracy",
        "s008_accuracy",
        "s009_accuracy",
    ],
    variance_threshold=None,
    prune_threshold=None,
    patience=None,
)